In [30]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import scipy
from scipy import stats
import seaborn as sns
from functools import reduce
import pathlib
from csv import reader
%matplotlib inline
import warnings
import xlsxwriter
from sklearn.decomposition import PCA
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.stats import pearsonr
warnings.filterwarnings('ignore')

In [31]:
#provide folder paths
input_folder_path = '/Users/nropek/Dropbox (Dropbox @RU)/TurboID manuscript/Mass-spectrometry datasets/TurboID_analysis/notebook_analysis/02_ROC_analysis_after_102622_meeting_feedback/min_1_peptide_per_protein/merge_100A_for_comparison/results_to_merge_100A'
output_path = '/Users/nropek/Dropbox (Dropbox @RU)/TurboID manuscript/Mass-spectrometry datasets/TurboID_analysis/notebook_analysis/02_ROC_analysis_after_102622_meeting_feedback/min_1_peptide_per_protein/merge_100A_for_comparison'

In [32]:
#turn path into object 
input_path_obj = pathlib.Path(input_folder_path)
#get absolute path
input_path_obj = input_path_obj.resolve()

output_path_obj = pathlib.Path(output_path)
#get absolute path
output_path_obj = output_path_obj.resolve()

In [33]:
list_of_file_paths = list(input_path_obj.iterdir())

list_to_drop = ['median_R(iWAT_HF)',
       'median_R(iWAT_chow)', 'median_SI(iWAT_HF)', 'median_SI(iWAT_chow)',
       'alias_uniprot_id', 'description', 'db', 'entry_name', 'Reviewed',
       'Entry Name', 'Protein names', 'Gene Names', 'Organism', 'Length',
       'Gene Ontology (biological process)', 'Gene Ontology (GO)',
       'Gene Ontology IDs', 'Gene Ontology (cellular component)',
       'Gene Ontology (molecular function)', 'Subcellular location [CC]',
       'Protein families', 'Signal peptide', 'Zinc finger', 'Mass', 'keratin',
       'TP', 'FP', 'liver', 'normal-tissue', 'adipose-tissue',
       'spleen-white-pulp', 'TPR-FPR_iWAT_HF_cre+_1/iWAT_HF_cre-_1',
       'TPR-FPR_iWAT_HF_cre+_1/iWAT_HF_cre-_2',
       'TPR-FPR_iWAT_HF_cre+_2/iWAT_HF_cre-_1',
       'TPR-FPR_iWAT_HF_cre+_2/iWAT_HF_cre-_2',
       'TPR-FPR_iWAT_HF_cre+_3/iWAT_HF_cre-_1',
       'TPR-FPR_iWAT_HF_cre+_3/iWAT_HF_cre-_2',
       'TPR-FPR_iWAT_chow_cre+_1/iWAT_chow_cre-_1',
       'TPR-FPR_iWAT_chow_cre+_1/iWAT_chow_cre-_2',
       'TPR-FPR_iWAT_chow_cre+_2/iWAT_chow_cre-_1',
       'TPR-FPR_iWAT_chow_cre+_2/iWAT_chow_cre-_2',
       'TPR-FPR_iWAT_chow_cre+_3/iWAT_chow_cre-_1',
       'TPR-FPR_iWAT_chow_cre+_3/iWAT_chow_cre-_2',
       'index', 'pep_num']

In [36]:
list_of_dfs = []
for file_path in list_of_file_paths:
    file_name = file_path.stem
    df = pd.read_excel(file_path, engine="openpyxl", sheet_name='ratio_raw_values') 
    df = df.set_index(['uniprot_id', 'annotation'])
    df = df.drop(list_to_drop, axis = 1)
    df = df.add_suffix("_"+file_name)
    df = df.reset_index()
    list_of_dfs.append(df)


In [37]:
merged_100A = reduce(lambda df1,df2: pd.merge(df1,df2,on=["uniprot_id", "annotation"], how="outer"), list_of_dfs)


In [38]:
merged_100A.columns



Index(['uniprot_id', 'annotation',
       'ratio_iWAT_HF_cre+_1/iWAT_HF_cre-_1_03_sum_per_protein',
       'ratio_iWAT_HF_cre+_1/iWAT_HF_cre-_2_03_sum_per_protein',
       'ratio_iWAT_HF_cre+_2/iWAT_HF_cre-_1_03_sum_per_protein',
       'ratio_iWAT_HF_cre+_2/iWAT_HF_cre-_2_03_sum_per_protein',
       'ratio_iWAT_HF_cre+_3/iWAT_HF_cre-_1_03_sum_per_protein',
       'ratio_iWAT_HF_cre+_3/iWAT_HF_cre-_2_03_sum_per_protein',
       'ratio_iWAT_chow_cre+_1/iWAT_chow_cre-_1_03_sum_per_protein',
       'ratio_iWAT_chow_cre+_1/iWAT_chow_cre-_2_03_sum_per_protein',
       ...
       'pass_cutoff_iWAT_chow_cre+_1/iWAT_chow_cre-_2_05_median_per_protein',
       'pass_cutoff_iWAT_chow_cre+_2/iWAT_chow_cre-_1_05_median_per_protein',
       'pass_cutoff_iWAT_chow_cre+_2/iWAT_chow_cre-_2_05_median_per_protein',
       'pass_cutoff_iWAT_chow_cre+_3/iWAT_chow_cre-_1_05_median_per_protein',
       'pass_cutoff_iWAT_chow_cre+_3/iWAT_chow_cre-_2_05_median_per_protein',
       'iWAT_HF_cre+_pass_cutoff_sum

In [42]:
merged_100A.to_csv(output_path_obj / ("merged_100A_raw_and_norm.csv"))